In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE

# 1. 데이터 로드 및 v4.2 피처 생성
df = pd.read_csv("AUS_v2_Ready_To_Train.csv")

# [v4.2 로직 재현] 상호작용 피처 생성
df['FE_DEBT_STRESS'] = df['DEBT_RATIO'] / (df['INTEREST_COVERAGE_RATIO'] + 1)
df['FE_CASH_VELOCITY'] = df['CASH_RATIO'] / (np.log1p(df['SALES_REVENUE']) + 1)
df['FE_OPS_BUFFER'] = (df['OPERATING_MARGIN'] / 100) * df['CASH_RATIO']

# Z-Score 정규화
cols_to_norm = ['CASH_RATIO', 'FE_LIQUIDITY_STRESS', 'FE_DEBT_STRESS', 'FE_CASH_VELOCITY', 'FE_OPS_BUFFER']
for col in cols_to_norm:
    df[f'Z_{col}'] = (df[col] - df[col].mean()) / (df[col].std() + 1e-6)

# 학습 준비
leakage_cols = ['MORATORIUM_COUNT', 'MORATORIUM_OVERDUE_AMOUNT', 'ACCOUNT_SUSPENSION_COUNT', 'CARD_ACCOUNT_COUNT']
exclude_cols = ['COMPANY_ID', 'COMPANY_ID_NORM', 'TARGET_Y', 'TMP_Y'] + leakage_cols
features = [c for c in df.columns if c not in exclude_cols]

X = df[features]
y = df['TARGET_Y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ---------------------------------------------------------
# 🧪 전략 1: Recursive Feature Elimination (RFE)
# ---------------------------------------------------------
print("🔎 [전략 1] RFE 최정예 피처 선발 중...")
base_estimator = RandomForestClassifier(n_estimators=100, max_depth=9, class_weight='balanced', random_state=42)
selector = RFE(base_estimator, n_features_to_select=12, step=1) # 12개 정예 선발
selector = selector.fit(X_train, y_train)

X_train_rfe = X_train.iloc[:, selector.support_]
X_test_rfe = X_test.iloc[:, selector.support_]

model_rfe = RandomForestClassifier(n_estimators=500, max_depth=9, class_weight='balanced', random_state=42)
model_rfe.fit(X_train_rfe, y_train)
auc_rfe = roc_auc_score(y_test, model_rfe.predict_proba(X_test_rfe)[:, 1])

# ---------------------------------------------------------
# 🧪 전략 2: Pseudo-Labeling (준지도 학습)
# ---------------------------------------------------------
print("🔥 [전략 2] Pseudo-Labeling 정답 확장 중...")
# v4.2 원본 모델로 예측 확률 산출
model_v42 = RandomForestClassifier(n_estimators=500, max_depth=9, class_weight='balanced', random_state=42)
model_v42.fit(X_train, y_train)
probs_all = model_v42.predict_proba(X)[:, 1]

# 확률 0.85 이상을 '부실'로 간주하여 데이터 증강
pseudo_threshold = 0.85
pseudo_idx = np.where((probs_all > pseudo_threshold) & (y == 0))[0] # 정상인데 AI가 부실이라 확신하는 놈들

X_pseudo = X.iloc[pseudo_idx]
y_pseudo = pd.Series([1] * len(pseudo_idx))

X_train_pl = pd.concat([X_train, X_pseudo])
y_train_pl = pd.concat([y_train, y_pseudo])

model_pl = RandomForestClassifier(n_estimators=500, max_depth=9, class_weight='balanced', random_state=42)
model_pl.fit(X_train_pl, y_train_pl)
auc_pl = roc_auc_score(y_test, model_pl.predict_proba(X_test)[:, 1])

# ---------------------------------------------------------
# 📊 결과 보고
# ---------------------------------------------------------
print("\n" + "="*50)
print(f"✅ Baseline (v4.2) AUC: 0.7256")
print(f"🚀 [전략 1] RFE 적용 후 AUC: {auc_rfe:.4f}")
print(f"🔥 [전략 2] Pseudo-Labeling 적용 후 AUC: {auc_pl:.4f}")
print("="*50)

if auc_rfe > 0.7256:
    print(f"📌 추천: 피처 수를 {len(X_train_rfe.columns)}개로 줄이는 것이 유리합니다.")
    print(f"선택된 피처: {list(X_train_rfe.columns)}")

🔎 [전략 1] RFE 최정예 피처 선발 중...
🔥 [전략 2] Pseudo-Labeling 정답 확장 중...


C:\Users\cozy1\AppData\Local\Temp\ipykernel_47280\2620254631.py:63: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  y_train_pl = pd.concat([y_train, y_pseudo])



✅ Baseline (v4.2) AUC: 0.7256
🚀 [전략 1] RFE 적용 후 AUC: 0.6887
🔥 [전략 2] Pseudo-Labeling 적용 후 AUC: 0.7256


In [2]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss

# 1. Pseudo-Labeling 재시도 (문턱을 0.70으로 하향)
pseudo_threshold = 0.70 # 0.85에서 하향
pseudo_idx = np.where((probs_all > pseudo_threshold) & (y == 0))[0]

if len(pseudo_idx) > 0:
    X_train_pl = pd.concat([X_train, X.iloc[pseudo_idx]])
    y_train_pl = pd.concat([y_train, pd.Series([1] * len(pseudo_idx))])
    print(f"✅ {len(pseudo_idx)}개의 가짜 정답(Pseudo-labels) 확보!")
else:
    X_train_pl, y_train_pl = X_train, y_train
    print("⚠️ 여전히 문턱을 넘는 데이터가 없습니다.")

# 2. 확률 보정 모델 (Calibrated Classifier) 적용
# 기존 v4.2 모델을 기반으로 Isotonic 보정 수행
base_model = RandomForestClassifier(n_estimators=500, max_depth=9, class_weight='balanced', random_state=42)
calibrated_model = CalibratedClassifierCV(base_model, method='isotonic', cv=3)
calibrated_model.fit(X_train_pl, y_train_pl)

# 3. 성능 평가
y_prob_cal = calibrated_model.predict_proba(X_test)[:, 1]
new_auc = roc_auc_score(y_test, y_prob_cal)
brier_score = brier_score_loss(y_test, y_prob_cal)

print("\n" + "="*50)
print(f"🚀 [전략 3] 확률 보정 및 PL 하향 후 AUC: {new_auc:.4f}")
print(f"📊 Brier Score (낮을수록 정확): {brier_score:.4f}")
print("="*50)

⚠️ 여전히 문턱을 넘는 데이터가 없습니다.

🚀 [전략 3] 확률 보정 및 PL 하향 후 AUC: 0.7352
📊 Brier Score (낮을수록 정확): 0.0347


In [3]:
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier # 설치되어 있지 않다면 pip install xgboost
from sklearn.linear_model import LogisticRegression

# 1. 개별 기본 모델 설정
base_models = [
    ('rf', RandomForestClassifier(n_estimators=500, max_depth=9, class_weight='balanced', random_state=42)),
    ('xgb', XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, scale_pos_weight=20, random_state=42))
]

# 2. 스태킹 앙상블 (최종 판단은 로지스틱 회귀가 수행)
stack_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(),
    cv=5,
    stack_method='predict_proba'
)

print("🚀 [최종 단계] 스태킹 앙상블 학습 시작...")
stack_model.fit(X_train, y_train)

# 3. 최종 성능 평가
y_prob_stack = stack_model.predict_proba(X_test)[:, 1]
final_auc = roc_auc_score(y_test, y_prob_stack)

print("\n" + "="*50)
print(f"🔥 [AUS v4.4] 최종 스태킹 AUC: {final_auc:.4f}")
print("="*50)

🚀 [최종 단계] 스태킹 앙상블 학습 시작...

🔥 [AUS v4.4] 최종 스태킹 AUC: 0.7340


In [4]:
import pandas as pd

# 1. 피처 그룹핑 및 명세 정의
feature_audit = {
    "그룹": [
        "유동성(Liquidity)", "유동성(Liquidity)", "안정성(Stability)", "수익성(Profitability)", 
        "규모/활동성(Size)", "공급망(Network)", "복합 파생(Interaction)", "복합 파생(Interaction)", 
        "복합 파생(Interaction)", "상대적 지표(Z-Score)", "상대적 지표(Z-Score)"
    ],
    "피처명": [
        "CASH_RATIO", "FE_LIQUIDITY_STRESS", "DEBT_RATIO", "OPERATING_MARGIN",
        "FE_LOG_EMPLOYEE", "LINKED_PARTNERS", "FE_DEBT_STRESS", "FE_CASH_VELOCITY",
        "FE_OPS_BUFFER", "Z_CASH_RATIO", "Z_FE_CASH_VELOCITY"
    ],
    "계산 로직 (Formula)": [
        "현금 / 유동부채", "(현금 + 예치금) / 총자산 대비 가중치", "총부채 / 자기자본", "영업이익 / 매출액",
        "log1p(직원 수)", "연결된 거래처 수 (내부 DB)", "부채비율 / (이자보상배율 + 1)", "현금비율 / log1p(매출액)",
        "(영업이익률 / 100) * 현금비율", "(x - μ) / σ (전체 기업 대비)", "(x - μ) / σ (전체 기업 대비)"
    ],
    "비즈니스 의미 (Insight)": [
        "단기 채무 지급 능력", "자금 고갈에 대한 스트레스 저항력", "장기적 재무 건전성", "영업 활동의 효율성",
        "기업 규모의 로그 보정 (아웃라이어 제거)", "공급망 내 고립도 및 의존도", "원리금 상환 압박 중첩 리스크", 
        "매출 규모 대비 자금 회전의 건강도", "수익성과 유동성의 동시 악화 포착", "동종 업계 대비 현금 보유 수준", "업계 평균 대비 자금 흐름 속도"
    ]
}

df_audit = pd.DataFrame(feature_audit)

# 2. 출력 및 시각화용 데이터 정리
print("📊 [AUS v4.3] 피처 엔지니어링 명세서")
print("-" * 110)
pd.set_option('display.max_colwidth', None)
display(df_audit)

# 3. (Optional) 현재 모델에서의 기여도와 매칭 (v4.2/4.3 중요도 기반)
# feature_importances_ 필드를 사용하여 어떤 카테고리가 강세인지 분석 가능합니다.

📊 [AUS v4.3] 피처 엔지니어링 명세서
--------------------------------------------------------------------------------------------------------------


,그룹,피처명,계산 로직 (Formula),비즈니스 의미 (Insight)
0,유동성(Liquidity),CASH_RATIO,현금 / 유동부채,단기 채무 지급 능력
1,유동성(Liquidity),FE_LIQUIDITY_STRESS,(현금 + 예치금) / 총자산 대비 가중치,자금 고갈에 대한 스트레스 저항력
2,안정성(Stability),DEBT_RATIO,총부채 / 자기자본,장기적 재무 건전성
3,수익성(Profitability),OPERATING_MARGIN,영업이익 / 매출액,영업 활동의 효율성
4,규모/활동성(Size),FE_LOG_EMPLOYEE,log1p(직원 수),기업 규모의 로그 보정 (아웃라이어 제거)
5,공급망(Network),LINKED_PARTNERS,연결된 거래처 수 (내부 DB),공급망 내 고립도 및 의존도
6,복합 파생(Interaction),FE_DEBT_STRESS,부채비율 / (이자보상배율 + 1),원리금 상환 압박 중첩 리스크
7,복합 파생(Interaction),FE_CASH_VELOCITY,현금비율 / log1p(매출액),매출 규모 대비 자금 회전의 건강도
8,복합 파생(Interaction),FE_OPS_BUFFER,(영업이익률 / 100) * 현금비율,수익성과 유동성의 동시 악화 포착
9,상대적 지표(Z-Score),Z_CASH_RATIO,(x - μ) / σ (전체 기업 대비),동종 업계 대비 현금 보유 수준
